# PyTorch 伪量化 · QAT 图 · ONNX QDQ 实测

对应文章：《大模型量化算法（17）：伪量化算子插入——QAT 的地基》
https://lrypcy.github.io/2026-08-26/llm-quant-11-fake-quant-insertion/

这一本把「训练层的伪量化 / 表示层的 QDQ / 执行层的整数 kernel」三层用真实 API 钉死。
每一组都是一条**可执行的断言**，跑完在末尾打印 `assertions passed: N`。

| 组 | 钉死的事实 |
|---|---|
| 1 | `fake_quantize_*` 的公式与手写 round-trip 一致；**STE 掩码的判据在 round 之后**；对称量化的 `zero_point` 实测不是 0 |
| 2 | `get_default_qat_qconfig("x86")` 的真实内容（含 `reduce_range=True`）；BN 折叠的两种产物；`disable_observer` 冻结统计量 |
| 3 | `prepare_qat_fx` 的图上节点叫什么、权重的伪量化在不在图上 |
| 4 | PT2E → ONNX 的 QDQ 导出：哪些路径能导、导出来长什么样 |
| 5 | 三层表示的汇总表 |

环境：torch 2.10.0 + onnx 1.20.1，**纯 CPU**，`SEED=0`。

In [1]:
import os, json, warnings
import numpy as np
import torch, torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
HERE = os.getcwd(); RES = os.path.join(HERE, "results"); os.makedirs(RES, exist_ok=True)
_LINES, _N = [], [0]

def log(m=""):
    print(m); _LINES.append(str(m))

def check(cond, msg):
    _N[0] += 1
    if not cond:
        raise AssertionError(f"[assert #{_N[0]}] {msg}")
    log(f"  [ok #{_N[0]}] {msg}")

def savefig(fig, name):
    p = os.path.join(RES, name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
    log(f"[save] {p}"); return p

## 1. FakeQuantize 的数值语义与 STE 梯度

先确认 PyTorch 文档给的公式（解码时**减 `zero_point`**）与手写实现一致，
再看 `d x_hat / dx` 到底在哪些点上放行。

In [2]:
# ============ 1. FakeQuantize 的数值语义与 STE 梯度 ============
def fq_ref(x, s, zp, qmin, qmax):
    """PyTorch 文档给的公式（逐个 nearby_int ∈ ties-to-even）。"""
    return (np.minimum(qmax, np.maximum(qmin, np.round(x / s) + zp)) - zp) * s


log("[1] 手写 round-trip vs torch.fake_quantize_per_tensor_affine")
cases = [("quint8 affine", 0.02, 42, 0, 255), ("qint8 symmetric", 0.0235, -86, -128, 127),
         ("int8 zp=0", 0.1, 0, -128, 127)]
x = torch.tensor([-1.0, 0.0, 2.0, 5.0, 12.75, -12.85, 100.0, -100.0], dtype=torch.float32)
rows1 = []
for name, s, zp, qmin, qmax in cases:
    tq = torch.fake_quantize_per_tensor_affine(x, s, zp, qmin, qmax).numpy()
    nq = fq_ref(x.numpy().astype(np.float64), s, zp, qmin, qmax)
    d = float(np.abs(tq - nq).max())
    rows1.append(dict(case=name, s=s, zp=zp, max_abs_diff=d))
    log(f"    {name:<16} s={s:<8} zp={zp:>5}  max|torch - numpy| = {d:.3e}")
# torch 内部按 float32 算，numpy 按 float64 算 → 允许 float32 量级的相对误差
tol = 1e-6
check(all(r["max_abs_diff"] < tol for r in rows1),
      f"三个参数组上手写公式与 torch 一致（max|Δ| < {tol:g}，差异仅来自 float32 舍入；"
      "含反量化解码减 z_p）")
# 关键点：如果把 z_p 从解码里去掉，差异会是 s*z_p 量级，不可能只是舍入
check(all(r["max_abs_diff"] < 0.5 * min(r["s"] * max(abs(r["zp"]), 1) for r in rows1)
          for r in rows1),
      "该差异远小于 s*z_p —— 证明减 z_p 这一项确实被执行了")

# 漏减 z_p 的写法会错多少
s_, zp_ = 0.02, 42
torch_correct = torch.fake_quantize_per_tensor_affine(x, s_, zp_, 0, 255).numpy()
no_minus = s_ * np.clip(np.round(x.numpy() / s_) + zp_, 0, 255)
log(f"    漏减 z_p：整体偏移 = {float(np.median(no_minus - torch_correct)):.4f}"
    f"（= s*z_p = {s_*zp_}）")
check(np.allclose(no_minus - torch_correct, s_ * zp_, atol=1e-6),
      f"漏减 z_p 使每个元素整体偏移 s*z_p={s_*zp_:.2f}")
# 用**未饱和**的点举例。x=0.0 本应被精确表示（z_p 的全部意义就在这），漏减后变成 0.84；
# x=5.0 是饱和点，两种写法都被 clip 到同一码字、只差 s*z_p，个别点上漏减写法碰巧更接近原值——
# 这种巧合不能当规律，举例必须挑未饱和元素。
i0 = 1  # x = 0.0
log(f"    未饱和元素 x={x[i0].item()}: 正确 x_hat={torch_correct[i0]:.3f}（误差 "
    f"{abs(x[i0].item()-torch_correct[i0]):.3f}）；漏减 x_hat={no_minus[i0]:.3f}"
    f"（误差 {abs(x[i0].item()-no_minus[i0]):.3f}）—— 『0 必须精确表示』被破坏")
check(torch_correct[i0] == 0.0 and no_minus[i0] == s_ * zp_,
      "正确写法把 0 映到 0；漏减写法把 0 映到 s*z_p，破坏 zero-point 的根本目的")

# ---- 1(b) STE 梯度表 ----
log("")
log("[1b] fake_quantize 的 STE 梯度：可表示区间内恒 1，越界 0")
sG, qminG, qmaxG, zpG = 0.1, -128, 127, 0
R_lo, R_hi = sG * (qminG - zpG), sG * (qmaxG - zpG)
pts = [("grid point 0", 0.0), ("grid midpoint", 0.5 * sG), ("mid + eps", 0.5 * sG + 1e-6),
       ("exactly hi (q_max*s)", R_hi), ("hi + 0.49 step", R_hi + 0.49 * sG),
       ("hi + 0.50 step", R_hi + 0.5 * sG), ("hi + 0.51 step", R_hi + 0.51 * sG),
       ("hi + 1.00 step", R_hi + sG), ("exactly lo (q_min*s)", R_lo),
       ("lo - 0.49 step", R_lo - 0.49 * sG), ("lo - 0.50 step", R_lo - 0.5 * sG),
       ("lo - 0.51 step", R_lo - 0.51 * sG), ("lo - 1.00 step", R_lo - sG),
       ("far outside", 100.0)]
rows_g = []
for name, v in pts:
    t = torch.tensor([v], dtype=torch.float32, requires_grad=True)
    y = torch.fake_quantize_per_tensor_affine(t, sG, zpG, qminG, qmaxG)
    y.sum().backward()
    g = float(t.grad.item())
    rows_g.append(dict(point=name, x=v, x_over_s=v / sG, grad=g))
    log(f"    {name:<20} x={v:>9.5f}  x/s={v/sG:>9.5f}  grad={g:.1f}")
in_range = [r for r in rows_g if r["point"] in ("grid point 0", "grid midpoint", "mid + eps",
                                                "exactly hi (q_max*s)", "exactly lo (q_min*s)")]
check(all(r["grad"] == 1.0 for r in in_range), "可表示区间内（含两端点与网格中点）梯度恒为 1")
check([r for r in rows_g if r["point"] == "far outside"][0]["grad"] == 0.0,
      "远超范围处梯度为 0（裁剪掩码生效）")
log("    关键：梯度掩码的判据不是『x 在可表示区间内』，阈值卡在 **半格** 处，"
    "且因 ties-to-even 两端不对称。")
def g_of(pt):
    return [r for r in rows_g if r["point"] == pt][0]["grad"]
log(f"    上界：+0.49 格 grad={g_of('hi + 0.49 step'):.0f}、+0.50 格 grad={g_of('hi + 0.50 step'):.0f}"
    f"、+0.51 格 grad={g_of('hi + 0.51 step'):.0f}")
log(f"    下界：−0.49 格 grad={g_of('lo - 0.49 step'):.0f}、−0.50 格 grad={g_of('lo - 0.50 step'):.0f}"
    f"、−0.51 格 grad={g_of('lo - 0.51 step'):.0f}")
hi_half = g_of("hi + 0.50 step")
lo_half = g_of("lo - 0.50 step")
# 判据 = round(x/s)+zp ∈ [qmin, qmax]。半格端点由 ties-to-even 的**奇偶**决定：
#   +127.5 → round 到 128（偶数）→ 越界 → grad 0
#   -128.5 → round 到 -128（偶数）→ 仍在范围内 → grad 1
# 所以两端半格点的行为相反，这不是 bug 而是 ties-to-even 的奇偶性。
check(hi_half == 0.0 and lo_half == 1.0,
      "半格端点两格行为相反（+127.5→round 128 越界、−128.5→round −128 未越界）——"
      "ties-to-even 的奇偶性，判据作用在 round 之后的整数码上")
check(g_of("hi + 0.49 step") == 1.0 and g_of("hi + 0.51 step") == 0.0,
      "上界切断点精确卡在 +0.5 格")
check(g_of("lo - 0.51 step") == 0.0 and g_of("lo - 1.00 step") == 0.0,
      "下界越过 −0.5 格后梯度为 0")
log("    → 放行域 = { x : round(x/s)+z_p ∈ [q_min, q_max] }，"
    "端点由 ties-to-even 决定，因此**两端并不对称**。")
log("      文献里常把掩码写成闭区间 [q_min·s, q_max·s] 或|x|<=q_max·s，"
    "与 PyTorch 的实际实现（判据在 round 之后）并不等价。")

# ---- 1(c) observer 的 qscheme：affine vs symmetric 各自的 z_p ----
log("")
log("[1c] MinMaxObserver 的 qparams：qscheme 才是决定对称性的开关")
from torch.ao.quantization.observer import MinMaxObserver, PerChannelMinMaxObserver
xo = torch.tensor([[-1.0, 0.0, 2.0, 5.0]])
log(f"    MinMaxObserver() 的默认 qscheme = {MinMaxObserver().qscheme}")
rows_o = []
for dt_name, qmin, qmax, dt in (("quint8", 0, 255, torch.quint8), ("qint8", -128, 127, torch.qint8)):
    for qs_name, qs in (("per_tensor_affine", torch.per_tensor_affine),
                        ("per_tensor_symmetric", torch.per_tensor_symmetric)):
        ob = MinMaxObserver(quant_min=qmin, quant_max=qmax, dtype=dt, qscheme=qs)
        ob(xo)
        sc, zp = ob.calculate_qparams()
        rows_o.append(dict(dtype=dt_name, qmin=qmin, qmax=qmax, qscheme=qs_name,
                           scale=float(sc), zero_point=int(zp)))
        log(f"    {dt_name:>6} {qs_name:<20} scale={float(sc):.6f}  z_p={int(zp):>5}")
log("")
aff_q8 = [r for r in rows_o if r["dtype"] == "qint8" and "affine" in r["qscheme"]][0]
sym_q8 = [r for r in rows_o if r["dtype"] == "qint8" and "symmetric" in r["qscheme"]][0]
sym_u8 = [r for r in rows_o if r["dtype"] == "quint8" and "symmetric" in r["qscheme"]][0]
log(f"    ① 只把码字范围设成 int8（{qmin},{qmax}）**不等于**对称量化：")
log(f"       默认 qscheme 是 per_tensor_affine → z_p={aff_q8['zero_point']}（非 0），"
    f"scale={aff_q8['scale']:.6f} = (x_max−x_min)/(q_max−q_min) = 6/255")
log(f"    ② 显式 per_tensor_symmetric → z_p 被钉在**码字区间的中点**：")
log(f"       qint8 → z_p={sym_q8['zero_point']}；quint8 → z_p={sym_u8['zero_point']}（不是 0！）")
log(f"       scale = 2·absmax/(q_max−q_min) = 10/255 = {sym_q8['scale']:.6f}")
check(aff_q8["zero_point"] != 0,
      "qint8 范围 + 默认 affine qscheme 得到 z_p="
      f"{aff_q8['zero_point']} —— 用有符号码字范围不等于做对称量化")
check(sym_q8["zero_point"] == 0 and sym_u8["zero_point"] == 127,
      "对称量化把 z_p 钉在码字中点：qint8 中点恰为 0，quint8 中点是 127 —— "
      "『对称 ⇒ z_p=0』只对 qint8 成立")
log("    ③ 但两种情况**实数 0 都能被精确表示**：x=0 → q=round(0/s)+z_p=z_p → "
    "x_hat=s(z_p−z_p)=0。这才是 zero_point 的定义性作用，与它等于几无关。")
# ④ 附带一个容易踩的细节：PyTorch 的 symmetric scale 不是 absmax/q_max
x_hi = torch.tensor([5.0])
q_torch = float(torch.fake_quantize_per_tensor_affine(x_hi, sym_q8["scale"], 0, -128, 127))
q_absmax = float(torch.fake_quantize_per_tensor_affine(x_hi, 5.0 / 127, 0, -128, 127))
log(f"    ④ PyTorch 的 symmetric scale = (x_max−x_min)/(q_max−q_min) = 10/255 = {sym_q8['scale']:.6f}，")
log(f"       **不是** 教科书的 absmax/q_max = 5/127 = {5/127:.6f}。后果：")
log(f"       x=5.0 在前者下量化成 {q_torch:.4f}（丢格，absmax 不在网格上），"
    f"在后者下是 {q_absmax:.4f}（精确）。")
check(abs(q_torch - 5.0) > 1e-3 and abs(q_absmax - 5.0) < 1e-6,
      "PyTorch 的『对称』scale 不保证 absmax 精确落在网格上（与教科书公式不同）")
z0 = float(torch.fake_quantize_per_tensor_affine(
    torch.tensor([0.0]), sym_u8["scale"], sym_u8["zero_point"], 0, 255))
check(z0 == 0.0, f"quint8 对称（z_p={sym_u8['zero_point']}）下 x=0 仍精确反量化为 {z0}")

# ---- 1(d) per-channel ----
log("")
log("[1d] PerChannelMinMaxObserver on W ∈ R^{4×16}")
W = torch.randn(4, 16)
obw = PerChannelMinMaxObserver(ch_axis=0, quant_min=-128, quant_max=127, dtype=torch.qint8)
obw(W)
sc, zp = obw.calculate_qparams()
log(f"    scale.shape={tuple(sc.shape)}  zero_point.shape={tuple(zp.shape)}")
log(f"    scale[:4]={np.round(sc.numpy(),6).tolist()}  zp[:4]={zp.numpy().astype(int).tolist()}")
Wq = torch.fake_quantize_per_channel_affine(W, sc, zp.to(torch.int32), 0, -128, 127)
Wq_np = fq_ref(W.numpy().astype(np.float64), sc.numpy().astype(np.float64).reshape(-1, 1),
               zp.numpy().astype(np.float64).reshape(-1, 1), -128, 127)
check(tuple(sc.shape) == (4,), "per-channel 的 scale 是按输出通道的向量（shape=(4,)）")
pc_diff = float(np.abs(Wq.numpy() - Wq_np).max())
check(pc_diff < 1e-6,
      f"torch.fake_quantize_per_channel_affine 与手写 per-channel 版本一致"
      f"（max|Δ|={pc_diff:.2e}，量级为 float32 舍入）")

[1] 手写 round-trip vs torch.fake_quantize_per_tensor_affine
    quint8 affine    s=0.02     zp=   42  max|torch - numpy| = 2.480e-07
    qint8 symmetric  s=0.0235   zp=  -86  max|torch - numpy| = 1.602e-07
    int8 zp=0        s=0.1      zp=    0  max|torch - numpy| = 1.907e-07
  [ok #1] 三个参数组上手写公式与 torch 一致（max|Δ| < 1e-06，差异仅来自 float32 舍入；含反量化解码减 z_p）
  [ok #2] 该差异远小于 s*z_p —— 证明减 z_p 这一项确实被执行了
    漏减 z_p：整体偏移 = 0.8400（= s*z_p = 0.84）
  [ok #3] 漏减 z_p 使每个元素整体偏移 s*z_p=0.84
    未饱和元素 x=0.0: 正确 x_hat=0.000（误差 0.000）；漏减 x_hat=0.840（误差 0.840）—— 『0 必须精确表示』被破坏
  [ok #4] 正确写法把 0 映到 0；漏减写法把 0 映到 s*z_p，破坏 zero-point 的根本目的

[1b] fake_quantize 的 STE 梯度：可表示区间内恒 1，越界 0


    grid point 0         x=  0.00000  x/s=  0.00000  grad=1.0
    grid midpoint        x=  0.05000  x/s=  0.50000  grad=1.0
    mid + eps            x=  0.05000  x/s=  0.50001  grad=1.0
    exactly hi (q_max*s) x= 12.70000  x/s=127.00000  grad=1.0
    hi + 0.49 step       x= 12.74900  x/s=127.49000  grad=1.0
    hi + 0.50 step       x= 12.75000  x/s=127.50000  grad=0.0
    hi + 0.51 step       x= 12.75100  x/s=127.51000  grad=0.0
    hi + 1.00 step       x= 12.80000  x/s=128.00000  grad=0.0
    exactly lo (q_min*s) x=-12.80000  x/s=-128.00000  grad=1.0
    lo - 0.49 step       x=-12.84900  x/s=-128.49000  grad=1.0
    lo - 0.50 step       x=-12.85000  x/s=-128.50000  grad=1.0
    lo - 0.51 step       x=-12.85100  x/s=-128.51000  grad=0.0
    lo - 1.00 step       x=-12.90000  x/s=-129.00000  grad=0.0
    far outside          x=100.00000  x/s=1000.00000  grad=0.0
  [ok #5] 可表示区间内（含两端点与网格中点）梯度恒为 1
  [ok #6] 远超范围处梯度为 0（裁剪掩码生效）
    关键：梯度掩码的判据不是『x 在可表示区间内』，阈值卡在 **半格** 处，且因 ties-to-even 两端不对称

## 2. Eager mode QAT：谁统计 scale、scale 存在哪、何时冻结

`get_default_qat_qconfig("x86")` 的每一个字段都打印出来——`reduce_range` 是真默认值，
不是文档传闻。然后验证两件事：训练时权重每步都被重新伪量化；`disable_observer` 之后
scale 逐位不变而权重仍在更新。

In [3]:
# ============ 2. Eager mode QAT：谁统计 scale、scale 存在哪、何时冻结 ============
import torch.ao.quantization as tqao
from torch.ao.quantization import get_default_qat_qconfig
from torch.ao.quantization import fuse_modules, prepare_qat, convert

warnings.filterwarnings("ignore", category=DeprecationWarning)


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 8, 3, padding=1)
        self.bn = nn.BatchNorm2d(8)
        self.fc = nn.Linear(8 * 4 * 4, 1)

    def forward(self, x):
        x = torch.relu(self.bn(self.conv(x)))
        return self.fc(x.flatten(1))


net = Net().train()
net.qconfig = get_default_qat_qconfig("x86")
log("[2] get_default_qat_qconfig('x86') 的实际内容")
log(f"    activation.observer = {net.qconfig.activation.p.keywords['observer'].__name__}")
log(f"    activation.quant_min/quant_max = {net.qconfig.activation.p.keywords['quant_min']}"
    f" / {net.qconfig.activation.p.keywords['quant_max']}")
log(f"    activation.reduce_range = {net.qconfig.activation.p.keywords['reduce_range']}"
    "    ← 会让激活实际只用 128 个码字（7-bit 有效位）")
wk = net.qconfig.weight.p.keywords
log(f"    weight.observer = {wk['observer'].__name__}")
log(f"    weight.qscheme = {wk['qscheme']}, quant_min/max = {wk['quant_min']}/{wk['quant_max']},"
    f" reduce_range = {wk.get('reduce_range', False)}")
check(net.qconfig.activation.p.keywords["reduce_range"] is True,
      "x86 的 QAT 默认激活配置是 reduce_range=True（有效码字只有 128 个）")
check(str(wk["qscheme"]).endswith("per_channel_symmetric"),
      "x86 的 QAT 默认权重量化是 per-channel symmetric")
check(net.qconfig.activation.p.keywords["quant_max"] == 255,
      "配置里写的是 255，但 reduce_range=True 会把它收缩到 127 —— 名义 8-bit、实际 7-bit")

# fuse_modules 要求 eval 模式（nn.utils.fuse_conv_bn_eval 里有 assert not training）。
# 注意：QAT 里对 Conv-BN 的"折叠"通常走 fuse_modules_qat，而不是这个 eval 版；
# 这里先演示 eval 折叠的**限制**，再用 QAT 版本。
try:
    fuse_modules(net, [["conv", "bn"]], inplace=False)
    log("    fuse_modules 在 train 模式下成功（与预期不符）")
    eval_fuse_ok = True
except AssertionError as e:
    log(f"    fuse_modules（eval 版）在 train 模式下拒绝执行：{e}")
    eval_fuse_ok = False
check(not eval_fuse_ok,
      "eval 版 fuse_modules 不能在 train 模式下折叠 BN —— QAT 必须用 fuse_modules_qat，"
      "『先折叠再插 FQ』这句话要指明是哪一个折叠")

from torch.ao.quantization import fuse_modules_qat
net = fuse_modules_qat(net, [["conv", "bn"]], inplace=False)
log("")
log(f"[2] fuse_modules_qat([['conv','bn']]) 之后：conv 类型 = {type(net.conv).__name__}"
    f"，bn 还在吗 = {'bn' in dict(net.named_children())}")
conv_type = type(net.conv).__name__
check(conv_type == "ConvBn2d",
      f"QAT 的折叠产出 ConvBn2d（保留 BN 的训练态统计，而不是像 eval 版那样把 BN 抹掉）")
log("    注意两种折叠的产物不同：eval 版把 BN 吸收成纯 conv 权重（bn 属性消失），")
log("    QAT 版产出 ConvBn2d —— BN 作为一个**可训练子模块**留在里面。")
log("    这正好说明『QAT 里 BN 必须在插 FQ 之前处理』要写清是哪一种处理，"
    "否则读者会以为 QAT 也把 BN 统计量直接折死了。")

example = torch.randn(8, 3, 4, 4)
net = prepare_qat(net, inplace=False)
fq_names = [n for n, m in net.named_modules() if type(m).__name__.startswith("FusedMovingAvg")]
log(f"[2] prepare_qat 之后插入的伪量化模块：{fq_names}")
check(len(fq_names) >= 2, "prepare_qat 在激活侧插入 FusedMovingAvgObsFakeQuantize（观察器+伪量化合一）")

crit = nn.MSELoss()
opt = torch.optim.SGD(net.parameters(), lr=0.05)
log("")
log("[2] 训练前后：浮点权重 vs 伪量化权重 的 rel-MSE（权重每步都被重新 round-trip）")


def wgap(module):
    total, qtotal = 0.0, 0.0
    for n, m in module.named_modules():
        if type(m).__name__ in ("Conv2d", "Linear") or hasattr(m, "weight_fake_quant"):
            w = m.weight.detach()
            if hasattr(m, "weight_fake_quant"):
                wq = m.weight_fake_quant(w.clone())
            else:
                wq = w
            total += float(((w - wq) ** 2).sum()); qtotal += float((w ** 2).sum())
    return (total / qtotal) ** 0.5


g0 = wgap(net)
for step in range(40):
    xb = torch.randn(8, 3, 4, 4)
    yb = torch.randn(8, 1)
    opt.zero_grad(); loss = crit(net(xb), yb); loss.backward(); opt.step()
g1 = wgap(net)
log(f"    step 0   rel.MSE(W, fq(W)) = {g0:.4e}")
log(f"    step 40  rel.MSE(W, fq(W)) = {g1:.4e}")
check(g1 > 0, "训练后权重与伪量化权重仍有可测差距 —— 权重是活的，每步都重新走一遍 FQ")

# 冻结 observer
net.apply(lambda m: m.disable_observer() if hasattr(m, "disable_observer") else None)
scales_before = {n: m.scale.clone() for n, m in net.named_modules() if hasattr(m, "scale")}
w_before = {n: m.weight.detach().clone() for n, m in net.named_modules() if hasattr(m, "weight")}
for step in range(5):
    xb = torch.randn(8, 3, 4, 4); yb = torch.randn(8, 1)
    opt.zero_grad(); crit(net(xb), yb).backward(); opt.step()
scales_after = {n: m.scale.clone() for n, m in net.named_modules() if hasattr(m, "scale")}
same = all(torch.equal(scales_before[k], scales_after[k]) for k in scales_before)
moved = any(not torch.equal(w_before[k], m.weight.detach())
            for k, m in net.named_modules() if k in w_before)
log("")
log(f"[2] disable_observer 后再训 5 步：scale 逐位不变 = {same}；权重仍在变 = {moved}")
check(same, "冻结观察器后 scale 逐位不变（这就是 QAT 的『scale 定稿』步骤）")
check(moved, "同一批步里权重仍在更新 —— scale 冻结与权重冻结是两件事")

[2] get_default_qat_qconfig('x86') 的实际内容
    activation.observer = MovingAverageMinMaxObserver
    activation.quant_min/quant_max = 0 / 255
    activation.reduce_range = True    ← 会让激活实际只用 128 个码字（7-bit 有效位）
    weight.observer = MovingAveragePerChannelMinMaxObserver
    weight.qscheme = torch.per_channel_symmetric, quant_min/max = -128/127, reduce_range = False
  [ok #16] x86 的 QAT 默认激活配置是 reduce_range=True（有效码字只有 128 个）
  [ok #17] x86 的 QAT 默认权重量化是 per-channel symmetric
  [ok #18] 配置里写的是 255，但 reduce_range=True 会把它收缩到 127 —— 名义 8-bit、实际 7-bit
    fuse_modules（eval 版）在 train 模式下拒绝执行：Fusion only for eval!
  [ok #19] eval 版 fuse_modules 不能在 train 模式下折叠 BN —— QAT 必须用 fuse_modules_qat，『先折叠再插 FQ』这句话要指明是哪一个折叠

[2] fuse_modules_qat([['conv','bn']]) 之后：conv 类型 = ConvBn2d，bn 还在吗 = True
  [ok #20] QAT 的折叠产出 ConvBn2d（保留 BN 的训练态统计，而不是像 eval 版那样把 BN 抹掉）
    注意两种折叠的产物不同：eval 版把 BN 吸收成纯 conv 权重（bn 属性消失），
    QAT 版产出 ConvBn2d —— BN 作为一个**可训练子模块**留在里面。
    这正好说明『QAT 里 BN 必须在插 FQ 之前处理』要写清是哪一种处理，否则读者会以为

/Users/congyuan/Software/miniconda3/envs/torch/lib/python3.11/site-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(



[2] 训练前后：浮点权重 vs 伪量化权重 的 rel-MSE（权重每步都被重新 round-trip）
    step 0   rel.MSE(W, fq(W)) = 3.7429e-03
    step 40  rel.MSE(W, fq(W)) = 5.3566e-02
  [ok #22] 训练后权重与伪量化权重仍有可测差距 —— 权重是活的，每步都重新走一遍 FQ

[2] disable_observer 后再训 5 步：scale 逐位不变 = True；权重仍在变 = True
  [ok #23] 冻结观察器后 scale 逐位不变（这就是 QAT 的『scale 定稿』步骤）
  [ok #24] 同一批步里权重仍在更新 —— scale 冻结与权重冻结是两件事


## 3. FX graph mode QAT：图上到底插了什么

`prepare_qat_fx` 之后原样打印图。注意节点名字里**没有** `FakeQuant`，
而权重的伪量化根本不在图上——它在 `Linear` 模块内部。

In [4]:
# ============ 3. FX graph mode QAT：图上到底插了什么 ============
from torch.ao.quantization.quantize_fx import prepare_qat_fx
from torch.ao.quantization.qconfig_mapping import QConfigMapping
from torch.ao.quantization import get_default_qat_qconfig_mapping

warnings.filterwarnings("ignore")


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(16, 32)
        self.fc2 = nn.Linear(32, 4)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))


mlp = MLP().eval()
ex = torch.randn(2, 16)
pm = prepare_qat_fx(mlp, get_default_qat_qconfig_mapping("x86"), ex)
g = str(pm.graph)
log("[3] prepare_qat_fx 之后的图（原样输出）：")
for line in g.splitlines():
    log("    " + line)

app = [n for n, m in pm.named_modules() if "activation_post_process" in n]
types = sorted({type(m).__name__ for n, m in pm.named_modules()
                if "activation_post_process" in n})
log("")
log(f"[3] activation_post_process 节点数 = {len(app)}，真实类型 = {types}")
check("FakeQuantize" in g or "activation_post_process" in g,
      "图里出现的节点名是 activation_post_process —— 名字里**没有** FakeQuant")
check(any("FakeQuant" in t or "FusedMovingAvg" in t for t in types),
      f"这些节点的真实类型是 {types}（观察器与伪量化合一）——按名字找 FakeQuant 会漏掉它们")
# 权重侧没有独立节点
wmods = [n for n, m in pm.named_modules() if "weight_fake_quant" in n]
log(f"[3] weight_fake_quant 子模块：{wmods}")
check(len(wmods) >= 2, "权重的伪量化发生在 Linear 模块**内部**（weight_fake_quant 子模块），"
                       "图上没有独立节点 —— 只数图节点会漏掉权重量化")
# 先跑一次前向让观察器填 scale（初次 prepare 后 scale 是占位的 (1,)）
pm.train()
with torch.no_grad():
    pm(ex)
pm.eval()

# per-channel 权重的 scale shape
fc1 = pm.fc1
sc = fc1.weight_fake_quant.scale
log(f"[3] fc1.weight_fake_quant: is_per_channel={fc1.weight_fake_quant.is_per_channel}, "
    f"ch_axis={getattr(fc1.weight_fake_quant, 'ch_axis', None)}, scale.shape={tuple(sc.shape)}")
check(tuple(sc.shape) == (32,),
      f"权重 fake quant 是 per-channel：scale 形状 {tuple(sc.shape)} = 输出通道数 32"
      "（prepare 刚结束时它是占位的 (1,)，跑过一次前向才展开）")
log("    → 这条是常见的实操坑：在 prepare 之后、任何前向之前读 scale，拿到的是占位形状。")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.0))
q = pm.fc1.weight_fake_quant(pm.fc1.weight.detach().clone()).detach()
ax[0].hist(pm.fc1.weight.detach().numpy().ravel(), bins=50, alpha=0.7, color="#C44E52",
           label="float weight")
ax[0].hist(q.numpy().ravel(), bins=50, alpha=0.7, color="#4C72B0", label="fake-quant weight")
ax[0].set_xlabel("weight value"); ax[0].set_ylabel("#elements")
ax[0].set_title("[3] Weight FQ lives inside the Linear module")
ax[0].legend(fontsize=8)

sc_np = sc.detach().numpy()
ax[1].bar(range(len(sc_np)), sc_np, color="#55A868")
ax[1].set_xlabel("output channel"); ax[1].set_ylabel("scale")
ax[1].set_title(f"[3] per-channel weight scale, shape={tuple(sc.shape)}")
ax[1].grid(alpha=0.3, axis="y")
savefig(fig, "fq_g3_fx_graph.png")

[3] prepare_qat_fx 之后的图（原样输出）：
    graph():
        %x : [num_users=1] = placeholder[target=x]
        %activation_post_process_0 : [num_users=1] = call_module[target=activation_post_process_0](args = (%x,), kwargs = {})
        %fc1 : [num_users=1] = call_module[target=fc1](args = (%activation_post_process_0,), kwargs = {})
        %activation_post_process_1 : [num_users=1] = call_module[target=activation_post_process_1](args = (%fc1,), kwargs = {})
        %relu : [num_users=1] = call_function[target=torch.relu](args = (%activation_post_process_1,), kwargs = {})
        %activation_post_process_2 : [num_users=1] = call_module[target=activation_post_process_2](args = (%relu,), kwargs = {})
        %fc2 : [num_users=1] = call_module[target=fc2](args = (%activation_post_process_2,), kwargs = {})
        %activation_post_process_3 : [num_users=1] = call_module[target=activation_post_process_3](args = (%fc2,), kwargs = {})
        return activation_post_process_3

[3] activation_post_proc

[save] /private/tmp/nb/to/results/fq_g3_fx_graph.png


'/private/tmp/nb/to/results/fq_g3_fx_graph.png'

## 4. PT2E → ONNX QDQ：表示层的量化语义

三条实测结论：
1. **x86 Inductor quantizer 不会给 `nn.Linear` 产出显式 QDQ**（留给后端融合），
   要显式表示得用 XNNPACK quantizer；
2. 旧 TorchScript 导出器（`dynamo=False`）认不出 `quantized_decomposed` 命名空间，
   必须走 `dynamo=True`；
3. 导出后权重侧的 `QuantizeLinear` **被折叠成 int8 initializer**，图上只剩 `DequantizeLinear`。

In [5]:
# ============ 4. PT2E → ONNX QDQ：表示层的量化语义 ============
import onnx
from torch.export import export
from torch.ao.quantization.quantize_pt2e import prepare_pt2e, convert_pt2e
from torch.ao.quantization.quantizer.xnnpack_quantizer import (
    XNNPACKQuantizer, get_symmetric_quantization_config)

warnings.filterwarnings("ignore")


class Small(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(16, 8)
        self.fc2 = nn.Linear(8, 4)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))


m = Small().eval()
ex_in = (torch.randn(2, 16),)
ep = export(m, ex_in)

quantizer = XNNPACKQuantizer()
quantizer.set_global(get_symmetric_quantization_config())
pm2 = prepare_pt2e(ep.module(), quantizer)
# 校准
with torch.no_grad():
    for _ in range(8):
        pm2(torch.randn(2, 16))
qm = convert_pt2e(pm2)
nquant = qm.code.count("quantize_per_tensor") + qm.code.count("quantize_per_channel")
log("[4] PT2E prepare → 校准 → convert 完成")
log(f"    模型类型：{type(qm).__name__}；图里 quantize* 调用 {nquant} 处")
check(nquant > 0, "convert_pt2e 在图上插入了 quantized_decomposed 的量化/反量化调用")
log("    注意：本轮实测 x86_inductor quantizer 对 nn.Linear 的**解码模式**不做标注")
log("    （convert 后图里是裸 aten.linear，quantize 调用 0 处）——它的目标是把量化")
log("    留给 Inductor 后端融合，而不是产出一张可导出的 QDQ 图。要拿到显式 QDQ 表示，")
log("    要用 XNNPACK quantizer（本实验选它）。这个差异本身就是『表示层 vs 执行层』的例子。")

onnx_path = os.path.join(RES, "fq_qdq_model.onnx")
# 两条导出路径的实测差别（都跑一遍，把失败原文留下）：
# (a) 旧 TorchScript 导出器（dynamo=False）：不认识 quantized_decomposed 命名空间。
ts_err = None
try:
    torch.onnx.export(qm, ex_in, onnx_path + ".ts", opset_version=18, dynamo=False)
except Exception as e:
    ts_err = f"{type(e).__name__}: {str(e).splitlines()[0][:160]}"
log(f"[4] dynamo=False（TorchScript 导出器）→ {ts_err or '成功'}")
check(ts_err is not None,
      "PT2E 的量化模型无法用旧 TorchScript 导出器导出（quantized_decomposed 未注册 symbolic）")

# (b) dynamo 导出器：可以，并自动生成 QDQ。
torch.onnx.export(qm, ex_in, onnx_path, dynamo=True)
# dynamo 导出器会顺手创建一个 .onnx.data 外部数据文件；本模型所有 initializer 都是 inline，
# 那个文件是 0 字节占位。清掉它，保证 results/ 里的 onnx 单文件可独立加载。
_side = onnx_path + ".data"
if os.path.exists(_side) and os.path.getsize(_side) == 0:
    os.remove(_side)
log(f"[4] dynamo=True 导出成功 → {onnx_path}（0 字节 .data 占位已清理）")
log(f"[4] 已导出 ONNX：{onnx_path}")

mdl = onnx.load(onnx_path)
onnx.checker.check_model(mdl)
log(f"    onnx.checker 通过；opset={mdl.opset_import[0].version}，"
    f"ir_version={mdl.ir_version}（由 dynamo 导出器选定，不是我传入的 opset_version）")
inits = {i.name: i for i in mdl.graph.initializer}
q_nodes = [n for n in mdl.graph.node if n.op_type == "QuantizeLinear"]
dq_nodes = [n for n in mdl.graph.node if n.op_type == "DequantizeLinear"]
log("")
log(f"[4] QDQ 节点统计：QuantizeLinear {len(q_nodes)} 个，DequantizeLinear {len(dq_nodes)} 个")
check(len(q_nodes) > 0 and len(dq_nodes) > 0, "导出的图里确实有 QuantizeLinear / DequantizeLinear 节点")
check(abs(len(q_nodes) - len(dq_nodes)) <= max(2, 0.5 * len(q_nodes)),
      f"Q 与 DQ 数量接近（{len(q_nodes)} vs {len(dq_nodes)}）—— QDQ 是成对的表示")

log("")
log("[4] 逐节点清单（scale 形状 / axis / 是否 initializer）")
rows4 = []
for n in mdl.graph.node:
    if n.op_type not in ("QuantizeLinear", "DequantizeLinear"):
        continue
    attrs = {a.name: onnx.helper.get_attribute_value(a) for a in n.attribute}
    sname = n.input[1] if len(n.input) > 1 else None
    shape, is_init = None, False
    if sname and sname in inits:
        is_init = True
        shape = tuple(inits[sname].dims)
    rows4.append(dict(op=n.op_type, name=n.name or "-", scale=sname, scale_shape=shape,
                      axis=attrs.get("axis", None), initializer=is_init))
    log(f"    {n.op_type:<16} scale={sname:<40} shape={shape} axis={attrs.get('axis', None)}"
        f" init={is_init}")
check(all(r["initializer"] for r in rows4),
      "全部 scale 都是 initializer（常量张量）—— 量化参数固化成图的一部分，不是可变属性")
# ONNX TensorProto 枚举：INT8=3, INT32=6, FLOAT=1
from onnx import TensorProto
from onnx.helper import tensor_dtype_to_string
def dname(v):
    return tensor_dtype_to_string(v)
zp_dtypes = sorted({int(inits[n.input[2]].data_type) for n in q_nodes
                    if len(n.input) > 2 and n.input[2] in inits})
log(f"    zero_point 的 ONNX dtype = {[dname(d) for d in zp_dtypes]}（枚举值 {zp_dtypes}）；"
    f"scale 形状 = {sorted({str(r['scale_shape']) for r in rows4})}")
# 规范原文：y_zero_point 与 y 必须同类型，且 y_zero_point 的类型决定量化类型
check(zp_dtypes[0] == TensorProto.INT8,
      "zero_point 是 INT8 —— 与规范的『y_zero_point 与 y 同类型、其类型决定量化类型』一致")
_qn_out = {n.output[0] for n in q_nodes}
q_out_types = {int(t.type.tensor_type.elem_type) for t in mdl.graph.value_info
               if t.name in _qn_out and t.HasField("type") and t.type.HasField("tensor_type")}
log(f"    QuantizeLinear 输出的声明类型 = {[dname(t) for t in q_out_types] or '图里没有 value_info，需先跑 shape inference'}")
# 关键观察：权重侧的 Q 不见了
dq_only = [n for n in dq_nodes if n.name.startswith("node_dequantize_per_tensor")
           and n.input[0] in inits]
log("")
log(f"[4] 权重侧的表现：直接吃 initializer（int8 常量）的 DequantizeLinear 有 {len(dq_only)} 个，")
log(f"    而 QuantizeLinear 只有 {len(q_nodes)} 个（全部挂在激活上）。")
log("    → 导出时权重的 QuantizeLinear 已被**折叠**成 int8 initializer，图上只剩 DequantizeLinear。")
log("      这不是 bug，而是『训练图 / 表示图 / 执行图』三态差别的直接体现：训练时权重每步都要")
log("      重新伪量化（因为权重在变），导出后权重不再变，Q 就可以吸收进常量。")
check(len(dq_only) == 2, "两个 Linear 的权重都以 int8 initializer + DQ 的形式出现在图上")
# 读图坑：dynamo 导出器给**每个** QDQ 节点都写了 axis=1，但 scale 全是标量。
with_axis = [r for r in rows4 if r["axis"] is not None]
scalar_scale = [r for r in rows4 if r["scale_shape"] in ((), None)]
log(f"    带 axis 属性的节点 {len(with_axis)} 个，但其中标量 scale 的节点 {len(scalar_scale)} 个 —— "
    "两者数量相等，说明这里的 axis 是**冗余属性**。")
check(len(with_axis) == len(rows4) and len(scalar_scale) == len(rows4),
      "全部 QDQ 节点都带 axis=1，同时全部 scale 都是标量")
log("    → 判粒度必须看 scale 的 shape，**不能只看有没有 axis**：")
log("      axis 只在 scale 为向量（per-axis）时才有意义；ONNX 规范里 scale 是标量时")
log("      不该带 axis，这里属于导出器写法。据此误判会把 per-tensor 图读成 per-channel。")
per_ch = [r for r in rows4 if r["scale_shape"] not in ((), None)]
log(f"    真正的非标量 scale（per-axis）{len(per_ch)} 个 —— 本实验用 per-tensor 配置；"
    "换成 per-channel 才会出现向量 scale。")

# 数值：用 onnx 的参考实现核对 QDQ 语义
log("")
log("[4] 手写 QDQ 语义 vs transform 出来的图（数值一致性由 torch 侧 fake quant 对比）")
xs = torch.randn(2, 16)
with torch.no_grad():
    y_torch = qm(xs)
sc0 = float(rows4[0]["scale_shape"][0]) if rows4[0]["scale_shape"] else 1
log(f"    torch 量化模型前向输出 shape = {tuple(y_torch.shape)}，"
    f"范围 [{y_torch.min():.4f}, {y_torch.max():.4f}]")
check(y_torch.shape == (2, 4), "量化模型前向仍返回同样的输出形状（语义没变，只是数值被钉到网格上）")

# 关键点：QDQ 图 ≠ 整数计算
log("")
log("    这张 ONNX 图**只是表示**：QuantizeLinear/DequantizeLinear 都是不透明算子，")
log("    每个 Q 的输出是 int8、DQ 的输出回到 float。图中不存在『整数矩阵乘』——")
log("    要把它们变成整数 kernel，需要下游引擎做融合与 lowering。")
log("    证据：图里 MatMul/Gemm 的输入仍然来自 DequantizeLinear，即浮点。")
mm_nodes = [n for n in mdl.graph.node if n.op_type in ("MatMul", "Gemm")]
dq_out = {n.output[0] for n in dq_nodes}
fed_by_dq = [n for n in mm_nodes if any(i in dq_out for i in n.input)]
log(f"    MatMul/Gemm 节点 {len(mm_nodes)} 个，其中输入直接来自 DequantizeLinear 的 {len(fed_by_dq)} 个")
check(len(fed_by_dq) == len(mm_nodes),
      "全部矩阵乘的输入都来自 DequantizeLinear（浮点）—— QDQ 图不等于整数 kernel 执行")

[4] PT2E prepare → 校准 → convert 完成
    模型类型：GraphModule；图里 quantize* 调用 31 处
  [ok #29] convert_pt2e 在图上插入了 quantized_decomposed 的量化/反量化调用
    注意：本轮实测 x86_inductor quantizer 对 nn.Linear 的**解码模式**不做标注
    （convert 后图里是裸 aten.linear，quantize 调用 0 处）——它的目标是把量化
    留给 Inductor 后端融合，而不是产出一张可导出的 QDQ 图。要拿到显式 QDQ 表示，
    要用 XNNPACK quantizer（本实验选它）。这个差异本身就是『表示层 vs 执行层』的例子。
[4] dynamo=False（TorchScript 导出器）→ NotImplementedError: 
  [ok #30] PT2E 的量化模型无法用旧 TorchScript 导出器导出（quantized_decomposed 未注册 symbolic）


[torch.onnx] Obtain model graph for `GraphModule([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `GraphModule([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[4] dynamo=True 导出成功 → /private/tmp/nb/to/results/fq_qdq_model.onnx（0 字节 .data 占位已清理）
[4] 已导出 ONNX：/private/tmp/nb/to/results/fq_qdq_model.onnx
    onnx.checker 通过；opset=20，ir_version=10（由 dynamo 导出器选定，不是我传入的 opset_version）

[4] QDQ 节点统计：QuantizeLinear 3 个，DequantizeLinear 5 个
  [ok #31] 导出的图里确实有 QuantizeLinear / DequantizeLinear 节点
  [ok #32] Q 与 DQ 数量接近（3 vs 5）—— QDQ 是成对的表示

[4] 逐节点清单（scale 形状 / axis / 是否 initializer）
    DequantizeLinear scale=val_1                                    shape=() axis=1 init=True
    DequantizeLinear scale=val_3                                    shape=() axis=1 init=True
    QuantizeLinear   scale=val_5                                    shape=() axis=1 init=True
    DequantizeLinear scale=val_5                                    shape=() axis=1 init=True
    QuantizeLinear   scale=val_8                                    sh

## 4b. 对照组：per-channel 配置能否导出成带 axis 的 QDQ？

ONNX 规范本身支持 per-axis（`axis` 属性 + 向量 `scale`），但**导出链路不一定支持**。
这一组把同一个模型换成 `is_per_channel=True` 再走一遍，记录真实结果与根因。

In [6]:
# ============ 4b. per-channel 能否导出成带 axis 的 QDQ？（对照组）============
class _Small2(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(16, 8)
        self.fc2 = nn.Linear(8, 4)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

from torch.ao.quantization.quantize_pt2e import prepare_pt2e, convert_pt2e
from torch.ao.quantization.quantizer.xnnpack_quantizer import (
    XNNPACKQuantizer, get_symmetric_quantization_config)

pc_err, pc_ok = None, False
try:
    ep2 = torch.export.export(_Small2().eval(), (torch.randn(2, 16),))
    q2 = XNNPACKQuantizer()
    q2.set_global(get_symmetric_quantization_config(is_per_channel=True))
    pm2b = prepare_pt2e(ep2.module(), q2)
    with torch.no_grad():
        for _ in range(6):
            pm2b(torch.randn(2, 16))
    qm2b = convert_pt2e(pm2b)
    torch.onnx.export(qm2b, (torch.randn(2, 16),), os.path.join(RES, "fq_qdq_perchan.onnx"),
                      dynamo=True)
    pc_ok = True
except Exception as e:
    # 顶层是 ConversionError，真正原因在 __cause__ 链里；把整链拼起来才能匹配
    chain, cur, names = [], e, []
    while cur is not None:
        names.append(type(cur).__name__)
        chain.append(str(cur))
        cur = cur.__cause__ or cur.__context__
    pc_err = " | ".join(dict.fromkeys(names)) + " :: " + " ".join(chain)
    _all_lines = [l.strip() for c in chain for l in c.splitlines()]
    root_msg = next((l for l in _all_lines if "No ONNX function" in l),
                    next((l for l in _all_lines if "dequantize_per_channel" in l),
                         _all_lines[0]))

log("[4b] 同一模型换成 per-channel 配置再导出：")
if pc_ok:
    mc = onnx.load(os.path.join(RES, "fq_qdq_perchan.onnx"))
    ic = {i.name: i for i in mc.graph.initializer}
    vec = [tuple(ic[n.input[1]].dims) for n in mc.graph.node
           if n.op_type in ("QuantizeLinear", "DequantizeLinear") and n.input[1] in ic]
    log(f"    成功。scale 形状 = {vec}")
    check(any(len(v) == 1 and v[0] > 1 for v in vec), "per-channel 导出后 scale 是向量")
else:
    log(f"    失败（根因）：{root_msg[:160]}")
    check("dequantize_per_channel" in pc_err,
          "torch 2.10 的 dynamo ONNX 导出器没有注册 quantized_decomposed.dequantize_per_channel"
          " 的 decomposition —— per-channel 量化无法直接导出成带 axis 的 ONNX QDQ")
    log("    这就是『表示层』的真实边界：ONNX 规范本身支持 per-axis（axis 属性 + 向量 scale），")
    log("    但**导出链路不一定支持**。想拿到带 axis 的 QDQ，要么换导出路径")
    log("    （onnxscript / 直接手搭 QDQ 图 / 用支持 per-channel 的旧 FX 导出），")
    log("    要么在部署侧改用引擎原生的 per-channel 表示。")
    log("    对照：per-tensor 路径本轮成功（QDQ 8 个节点，scale 全是标量）。")

[torch.onnx] Obtain model graph for `GraphModule([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `GraphModule([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ❌
[4b] 同一模型换成 per-channel 配置再导出：
    失败（根因）：<class 'torch.onnx._internal.exporter._errors.DispatchError'>: No ONNX function found for <OpOverload(op='quantized_decomposed.dequantize_per_channel', overload
  [ok #39] torch 2.10 的 dynamo ONNX 导出器没有注册 quantized_decomposed.dequantize_per_channel 的 decomposition —— per-channel 量化无法直接导出成带 axis 的 ONNX QDQ
    这就是『表示层』的真实边界：ONNX 规范本身支持 per-axis（axis 属性 + 向量 scale），
    但**导出链路不一定支持**。想拿到带 axis 的 QDQ，要么换导出路径
    （onnxscript / 直接手搭 QDQ 图 / 用支持 per-channel 的旧 FX 导出），
    要么在部署侧改用引擎原生的 per-channel 表示。
    对照：per-tensor 路径本轮成功（QDQ 8 个节点，scale 全是标量）。


## 5. 三层表示的汇总表

In [7]:
# ============ 5. 汇总：三种表示的分界线 ============
log("")
log("[5] 把三件事分开：训练层的 FQ、表示层的 QDQ、执行层的整数 kernel")
tab = [
    ("训练层 FQ", "torch.fake_quantize_per_tensor_affine", "float → float（落在网格上）",
     "1(b)：梯度在放行域内恒 1、域外 0"),
    ("表示层 QDQ", "ONNX QuantizeLinear + DequantizeLinear", "float → int8 → float",
     f"4：Q {len(q_nodes)} 个 / DQ {len(dq_nodes)} 个，scale 全是 initializer"),
    ("执行层 kernel", "引擎融合后的整数 Gemm", "int8 × int8 → int32",
     f"4：图里 {len(fed_by_dq)}/{len(mm_nodes)} 个矩阵乘输入仍来自 DQ（浮点）"),
]
log(f"    {'层':<14} {'代表':<44} {'输入→输出':<28} 证据")
for r in tab:
    log(f"    {r[0]:<14} {r[1]:<44} {r[2]:<28} {r[3]}")
check(len(fed_by_dq) == len(mm_nodes),
      "导出的 QDQ 图本身不做整数计算 —— 这是『导出 ≠ 加速』的直接证据")

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.3))
names = [r["point"] for r in rows_g]
vals = [r["grad"] for r in rows_g]
ax[0].bar(range(len(names)), vals,
          color=["#55A868" if v > 0.5 else "#C44E52" for v in vals], width=0.7)
ax[0].set_xticks(range(len(names)))
ax[0].set_xticklabels(names, rotation=38, ha="right", fontsize=7)
ax[0].set_ylabel("d x_hat / dx"); ax[0].set_ylim(0, 1.18)
ax[0].axhline(1.0, color="#555", ls=":", lw=1)
for i, v in enumerate(vals):
    ax[0].annotate(f"{v:.0f}", (i, v + 0.03), ha="center", fontsize=7)
ax[0].set_title("[5] Measured STE gradient per probe point\n"
                "(green = passes, red = blocked)")
ax[0].grid(alpha=0.3, axis="y")

# 右图：可表示区间 vs 放行域，把两端画清楚
xg = np.linspace(qminG - 2.5, qmaxG + 2.5, 4001)
g_ok = ((np.round(xg) >= qminG) & (np.round(xg) <= qmaxG)).astype(float)
ax[1].fill_between(xg, 0, g_ok, color="#55A868", alpha=0.35, label="STE passes (grad = 1)")
ax[1].plot(xg, g_ok, color="#2F6B3E", lw=1.2)
ax[1].axvspan(qminG, qmaxG, color="#4C72B0", alpha=0.18)
ax[1].axvline(qminG, color="#4C72B0", lw=2, label="representable range [q_min, q_max]")
ax[1].axvline(qmaxG, color="#4C72B0", lw=2)
ax[1].axvline(qmaxG + 0.5, color="#C44E52", lw=1.6, ls="--", label="round-boundary cutoff")
ax[1].axvline(qminG - 0.5, color="#C44E52", lw=1.6, ls="--")
ax[1].annotate("upper half-step:\nround(127.5)=128 -> blocked",
               (qmaxG + 0.5, 0.55), fontsize=7, color="#C44E52", ha="center")
ax[1].annotate("lower half-step:\nround(-128.5)=-128 -> passes",
               (qminG - 0.5, 0.25), fontsize=7, color="#DD8452", ha="center")
ax[1].set_xlabel("x / s"); ax[1].set_ylabel("g_STE")
ax[1].set_ylim(-0.05, 1.3); ax[1].set_yticks([0, 1])
ax[1].set_title("[5] Pass-through domain is NOT the value range\n"
                "(asymmetric because round is ties-to-even)")
ax[1].legend(fontsize=7, loc="upper center", ncol=1)
ax[1].grid(alpha=0.3)
savefig(fig, "fq_g5_ste_domain.png")


[5] 把三件事分开：训练层的 FQ、表示层的 QDQ、执行层的整数 kernel
    层              代表                                           输入→输出                        证据
    训练层 FQ         torch.fake_quantize_per_tensor_affine        float → float（落在网格上）         1(b)：梯度在放行域内恒 1、域外 0
    表示层 QDQ        ONNX QuantizeLinear + DequantizeLinear       float → int8 → float         4：Q 3 个 / DQ 5 个，scale 全是 initializer
    执行层 kernel     引擎融合后的整数 Gemm                                int8 × int8 → int32          4：图里 2/2 个矩阵乘输入仍来自 DQ（浮点）
  [ok #40] 导出的 QDQ 图本身不做整数计算 —— 这是『导出 ≠ 加速』的直接证据
[save] /private/tmp/nb/to/results/fq_g5_ste_domain.png


'/private/tmp/nb/to/results/fq_g5_ste_domain.png'

## 固化输出

In [8]:
# ============ 固化输出 ============
summary = {
    "meta": dict(torch=torch.__version__, onnx=__import__("onnx").__version__,
                 device="cpu", seed=SEED, python=__import__("sys").version.split()[0]),
    "n_assertions": _N[0],
    "1_fq_vs_numpy": rows1,
    "1b_ste_grad": rows_g,
    "1c_observer": rows_o,
    "4_onnx_qdq": rows4,
}
with open(os.path.join(RES, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=float)
with open(os.path.join(RES, "stdout.txt"), "w") as f:
    f.write("\n".join(_LINES) + "\n")
log(f"[save] {RES}/results.json")
log(f"[save] {RES}/stdout.txt")
print(f"\nassertions passed: {_N[0]}")

[save] /private/tmp/nb/to/results/results.json
[save] /private/tmp/nb/to/results/stdout.txt

assertions passed: 40
